# Chapter 11 Lab — Automatic Text Summarization

TextRank extractive summarization from scratch, scored with ROUGE, then compared against a
pretrained abstractive summarizer on the same small public-domain paragraphs.

## 1. TextRank from scratch

In [ ]:
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer

def textrank_summarize(sentences, top_k=2):
    X = TfidfVectorizer().fit_transform(sentences)
    sim = (X * X.T).toarray()
    graph = nx.from_numpy_array(sim)
    scores = nx.pagerank(graph)
    ranked = sorted(range(len(sentences)), key=lambda i: -scores[i])[:top_k]
    return [sentences[i] for i in sorted(ranked)]

document = (
    "The city council approved a new downtown park yesterday. "
    "The plan includes a playground, walking trails, and a small pond. "
    "Funding comes from a mix of city budget and a private donation. "
    "Construction is expected to begin next spring and finish within a year. "
    "Residents have long asked for more green space downtown."
)
sentences = [s.strip() for s in document.split(". ") if s.strip()]
summary = textrank_summarize(sentences, top_k=2)
print("\n".join(summary))

## 2. ROUGE evaluation against an author-written reference summary

In [ ]:
from rouge_score import rouge_scorer

reference = "The city approved a new downtown park with a playground and pond, funded by the city and a private donor, construction starting next spring."
candidate = " ".join(summary)

scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
print(scorer.score(reference, candidate))

## 3. Abstractive summarization for comparison

In [ ]:
from transformers import pipeline
summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-6-6")
abstractive = summarizer(document, max_length=40, min_length=15, do_sample=False)[0]["summary_text"]
print(abstractive)

## 4. A deliberate hallucination example

In [ ]:
tricky_doc = "The bakery on Elm Street closed on Friday after 12 years in business. The owner cited rising rent."
out = summarizer(tricky_doc, max_length=25, min_length=8, do_sample=False)[0]["summary_text"]
print("Source :", tricky_doc)
print("Summary:", out)
print("Check every claim in the summary against the source sentence by sentence — this is exactly\n"
      "the faithfulness spot-check habit Chapter 16 formalizes.")

## Exercise

Increase `top_k` in `textrank_summarize` to 3 and 4 and recompute ROUGE each time. Does ROUGE
keep improving as you extract more sentences? At what point does the "summary" stop doing its
job even if ROUGE goes up?